In [ ]:
# Permite importar src/ desde la carpeta notebooks/
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import DIR_DATA_RAW

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [ ]:
archivos = sorted((DIR_DATA_RAW / "telemetria").glob("*.parquet"))
df = pd.concat([pd.read_parquet(a) for a in archivos], ignore_index=True)

df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["ep"] = df["ep"].astype(str).str.strip().str.upper()

print(f"{len(df):,} filas | {df['ep'].nunique()} embarcaciones")
df.head()

In [ ]:
resumen = df.groupby("origen").agg(
    filas=("ep", "size"),
    embarcaciones=("ep", "nunique"),
    desde=("fecha", "min"),
    hasta=("fecha", "max"),
)
display(resumen)

print("\nNulos por columna (%):")
display((df.isna().mean() * 100).round(2).to_frame("% nulos").T)

In [ ]:
cols = ["Velocidad", "gal_h", "rpm"]
display(df.groupby("origen")[cols].describe(percentiles=[.05, .5, .95]).T)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, cols):
    sns.boxplot(data=df, x="origen", y=col, ax=ax, showfliers=False)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
# ¿Qué embarcaciones aparecen en qué fuente?
cruce = pd.crosstab(df["ep"], df["origen"])
display(cruce[(cruce > 0).sum(axis=1) > 1])  # las que están en más de una fuente

# ¿Registros repetidos (misma embarcación y mismo instante)?
dup = df.duplicated(subset=["ep", "fecha"], keep=False)
print(f"Registros duplicados por (ep, fecha): {dup.sum():,} ({dup.mean():.1%})")
display(df[dup].sort_values(["ep", "fecha"]).head(10))

In [ ]:
df["fecha"].min()

In [ ]:
q = df[cols].quantile([0.001, 0.01, 0.5, 0.99, 0.999])
display(q)

print(f"RPM >= 2000:   {(df['rpm'] >= 2000).sum():,} filas")
print(f"gal_h >= 300: {(df['gal_h'] >= 300).sum():,} filas")
print(f"Velocidad > 15 nudos: {(df['Velocidad'] > 15).sum():,} filas")
print(f"Velocidad < 0 o gal_h < 0: {((df['Velocidad'] < 0) | (df['gal_h'] < 0)).sum():,} filas")

In [ ]:
df["mes"] = df["fecha"].dt.to_period("M")
cobertura = df.pivot_table(index="ep", columns="mes", values="rpm", aggfunc="size", fill_value=0)

plt.figure(figsize=(16, 7))
sns.heatmap(np.log1p(cobertura), cmap="Blues", cbar_kws={"label": "log(registros)"})
plt.title("Registros por embarcación y mes")
plt.tight_layout()

display(df["ep"].value_counts().to_frame("registros"))